# 01 · Radar JSON Schema Inspection

**Project:** Physics-Informed Trigger Event Analysis — Urban Intersection Safety  
**Dataset:** USDOT Intersection Safety Challenge Stage-1B  
**Stage:** Task 1 — Radar JSON schema (no ML, no extraction)

---

## Purpose

Understand the internal structure of `Radars_Run_XXXX_sensorN.json` files
so we know exactly which fields are available for object-level motion analysis
and later physics-informed feature extraction.

## Key questions

- Is the radar JSON another MQTT message list (like the trigger JSON)?
- Where are object detections stored — inside `payload`?
- What fields describe each detection: id, class, speed, heading, position, lane, zone?
- Which timestamp should be used for alignment with trigger `receivedAt`?
- Are all 4 sensor files structurally identical?

## What this notebook does NOT do

- Does **not** extract any file to disk.
- Does **not** process all 800 runs — only `MAX_RUNS_TO_INSPECT` runs are sampled.
- Does **not** build features or models.
- Does **not** read video, LiDAR pcap, or SPaT pcap files.

## Outputs saved

| File | Description |
|------|-------------|
| `outputs/tables/radar_file_inventory.csv` | All radar JSON paths across all 4 zips |
| `outputs/tables/radar_schema_field_availability.csv` | Object field presence summary |
| `outputs/tables/radar_position_field_probe.csv` | Position field structure detail |
| `outputs/tables/radar_timestamp_probe.csv` | Timestamp field samples and format guesses |
| `outputs/tables/radar_sensor_schema_comparison.csv` | Per-sensor schema comparison |
| `outputs/tables/notebook_01_radar_schema_findings.md` | Auto-generated findings summary |

---
## 0 · Optional: Install / Verify Dependencies

In [3]:
# Uncomment if packages are missing in your Colab environment.
# !pip install -q pandas
print("[OK] Dependency check cell — uncomment pip install if needed.")

[OK] Dependency check cell — uncomment pip install if needed.


---
## 1 · Clone Repo from GitHub

**Edit `GITHUB_REPO_URL` to your repository before running.**

In [4]:
import os, subprocess

# ╔══════════════════════════════════════════════════════════════════╗
# ║  EDIT THIS URL                                                 ║
# ╠══════════════════════════════════════════════════════════════════╣
GITHUB_REPO_URL = "https://github.com/PulockDas/intersection_safety_trigger_project.git"
# ╚══════════════════════════════════════════════════════════════════╝

CLONE_DIR = "/content/intersection_safety_trigger_project"

if not os.path.exists(CLONE_DIR):
    print(f"[INFO] Cloning {GITHUB_REPO_URL} ...")
    result = subprocess.run(
        ["git", "clone", GITHUB_REPO_URL, CLONE_DIR],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("[OK]  Clone successful.")
    else:
        print("[ERROR]", result.stderr)
        raise RuntimeError("Git clone failed — check GITHUB_REPO_URL.")
else:
    print(f"[INFO] Repo already at {CLONE_DIR} — skipping clone.")

print("\n[INFO] Repo contents:")
for item in sorted(os.listdir(CLONE_DIR)):
    print(f"  {item}")

[INFO] Cloning https://github.com/PulockDas/intersection_safety_trigger_project.git ...
[OK]  Clone successful.

[INFO] Repo contents:
  .git
  .gitignore
  README.md
  notebooks
  outputs
  requirements.txt
  src


### 1.1 · Mount Google Drive (read-only — for training zip files)

In [5]:
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
    print("[INFO] Google Drive mounted — used ONLY to read training zip files.")
except ImportError:
    IN_COLAB = False
    print("[INFO] Not in Colab — Drive mount skipped.")

Mounted at /content/drive
[INFO] Google Drive mounted — used ONLY to read training zip files.


---
## 2 · Project Setup

`PROJECT_ROOT` is always the cloned repo — do not change it.  
Run **Cell 2.0** to auto-detect your training zip folder, then verify in **Cell 2.1**.

### 2.0 · Locate the Training Data Folder

In [6]:
import os
from pathlib import Path

# ╔══════════════════════════════════════════════════════════════════╗
# ║  Shared folder name in your Google Drive                       ║
# ╠══════════════════════════════════════════════════════════════════╣
FOLDER_NAME = "ITS_Intersection_USDOT"
# ╚══════════════════════════════════════════════════════════════════╝

DRIVE_ROOT   = Path("/content/drive")
MY_DRIVE     = DRIVE_ROOT / "MyDrive"
SHARED_DRIVE = DRIVE_ROOT / "Shareddrives"

print("Contents of /content/drive/:")
if DRIVE_ROOT.exists():
    for item in sorted(DRIVE_ROOT.iterdir()):
        print(f"  {item.name}/")
else:
    print("  [Drive not mounted — run Section 1.1 first]")

print("\nTop-level items in MyDrive/ (first 40):")
if MY_DRIVE.exists():
    for item in sorted(MY_DRIVE.iterdir())[:40]:
        print(f"  {item.name}{'/' if item.is_dir() else ''}")

print("\nShared Drives:")
if SHARED_DRIVE.exists():
    items = list(SHARED_DRIVE.iterdir())
    for item in sorted(items):
        print(f"  {item.name}/")
    if not items:
        print("  [none found]")
else:
    print("  [Shareddrives/ not present]")

candidates = [
    MY_DRIVE     / FOLDER_NAME,
    SHARED_DRIVE / FOLDER_NAME,
]

TRAINING_ZIP_DIR = None
print(f"\nSearching for '{FOLDER_NAME}' ...")
for c in candidates:
    try:
        if c.exists() and c.is_dir():
            zip_count = len(list(c.glob("*.zip")))
            print(f"  [FOUND] {c}  ({zip_count} zip file(s))")
            if zip_count > 0 and TRAINING_ZIP_DIR is None:
                TRAINING_ZIP_DIR = c
    except PermissionError:
        print(f"  [PERMISSION ERROR] {c}")

if TRAINING_ZIP_DIR:
    print(f"\n[OK] Auto-detected TRAINING_ZIP_DIR = {TRAINING_ZIP_DIR}")
else:
    print(f"\n[WARN] Could not auto-detect '{FOLDER_NAME}'.")
    print("       Set it manually in Cell 2.1.")

Contents of /content/drive/:
  .Encrypted/
  .Trash-0/
  .shortcut-targets-by-id/
  MyDrive/
  Othercomputers/

Top-level items in MyDrive/ (first 40):
  0_Introduction.gdoc
  2 Pdf_12_09_11_24_09.pdf
  2017831029.gdoc
  2017831036.gdoc
  2017831036_Verse.pptx
  2017831044_B.gdoc
  2017831046/
  2017831046 (1).pdf
  2017831046 (2).pdf
  2017831046 (3).pdf
  2017831046 (4).pdf
  2017831046 (5).pdf
  2017831046 (6).pdf
  2017831046 (7).pdf
  2017831046 (8).pdf
  2017831046.gdoc
  2017831046.pdf
  2017831046_A.pdf
  2017831046_B.pdf
  2017831046_Internship Report PSL.pdf
  20180914_115455.jpg
  20180914_115631.jpg
  20180914_115745.jpg
  20180914_115749.jpg
  20180914_132000.jpg
  20180914_132003.jpg
  20180914_132103.jpg
  20180914_132111.jpg
  20180914_133643.jpg
  20180914_145911.jpg
  20180914_151243.jpg
  20180914_151413 (1).jpg
  20180914_151413.jpg
  20181011_125902.jpg
  20190129_220734.jpg
  20190314_160847.jpg
  20190314_160849.jpg
  20190314_160852.jpg
  20190314_173758.jpg
  2

### 2.1 · Set Paths

If auto-detection above succeeded, this cell uses that result.  
If it failed, uncomment the manual override line.

In [7]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("/content/intersection_safety_trigger_project")

# ── Manual override (uncomment if auto-detection failed) ─────────────────────
# TRAINING_ZIP_DIR = Path("/content/drive/MyDrive/ITS_Intersection_USDOT")

if "TRAINING_ZIP_DIR" not in dir() or TRAINING_ZIP_DIR is None:
    raise RuntimeError(
        "TRAINING_ZIP_DIR is not set.\n"
        "Uncomment and edit the manual override line above."
    )

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
TABLES_DIR   = OUTPUTS_DIR  / "tables"
FIGURES_DIR  = OUTPUTS_DIR  / "figures"
SAMPLES_DIR  = OUTPUTS_DIR  / "samples"
LOGS_DIR     = OUTPUTS_DIR  / "logs"

print(f"PROJECT_ROOT     = {PROJECT_ROOT}")
print(f"SRC_DIR exists   = {SRC_DIR.exists()}")
print(f"TRAINING_ZIP_DIR = {TRAINING_ZIP_DIR}")
print(f"OUTPUTS_DIR      = {OUTPUTS_DIR}")

PROJECT_ROOT     = /content/intersection_safety_trigger_project
SRC_DIR exists   = True
TRAINING_ZIP_DIR = /content/drive/MyDrive/ITS_Intersection_USDOT
OUTPUTS_DIR      = /content/intersection_safety_trigger_project/outputs


### 2.2 · Inspection Parameters

Adjust these to control how deeply the notebook samples the data.

In [8]:
# ── How many runs to read radar JSON from ────────────────────────────────────
MAX_RUNS_TO_INSPECT            = 3

# ── How many sensor files to read per run (1–4) ──────────────────────────────
MAX_SENSOR_FILES_PER_RUN       = 4

# ── How many MQTT records to print per sensor file ───────────────────────────
MAX_RECORDS_PER_FILE_TO_PRINT  = 3

# ── How many detection objects to print per record ───────────────────────────
MAX_OBJECTS_PER_RECORD_TO_PRINT = 3

print(f"MAX_RUNS_TO_INSPECT            = {MAX_RUNS_TO_INSPECT}")
print(f"MAX_SENSOR_FILES_PER_RUN       = {MAX_SENSOR_FILES_PER_RUN}")
print(f"MAX_RECORDS_PER_FILE_TO_PRINT  = {MAX_RECORDS_PER_FILE_TO_PRINT}")
print(f"MAX_OBJECTS_PER_RECORD_TO_PRINT = {MAX_OBJECTS_PER_RECORD_TO_PRINT}")

MAX_RUNS_TO_INSPECT            = 3
MAX_SENSOR_FILES_PER_RUN       = 4
MAX_RECORDS_PER_FILE_TO_PRINT  = 3
MAX_OBJECTS_PER_RECORD_TO_PRINT = 3


### 2.3 · Imports

In [9]:
import json
import re
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
import pprint

import pandas as pd

# ── Project modules ──────────────────────────────────────────────────────────
from file_discovery import find_training_zips, make_output_dirs
from utils import format_size, check_mark, yes_no
from radar_parser import (
    find_radar_files_in_zip,
    safe_json_load_from_zip,
    decode_payload_if_needed,
    inspect_nested_schema,
    flatten_keys_sample,
    extract_object_list,
    get_object_list_field_name,
    check_object_fields,
    find_timestamp_fields,
    guess_timestamp_format,
    OBJECT_FIELDS_TO_CHECK,
    OBJECT_LIST_FIELDS,
    TIMESTAMP_FIELD_NAMES,
)

print("[OK] All imports successful.")

[OK] All imports successful.


### 2.4 · Create Output Directories & Locate Zip Files

In [10]:
make_output_dirs(TABLES_DIR, FIGURES_DIR, SAMPLES_DIR, LOGS_DIR)

zip_paths = find_training_zips(TRAINING_ZIP_DIR)
if not zip_paths:
    raise FileNotFoundError(
        f"No zip files found in {TRAINING_ZIP_DIR}.\n"
        "Check TRAINING_ZIP_DIR in Cell 2.1 and re-run."
    )

print("\nZip files found:")
for zp in zip_paths:
    print(f"  {zp.name}  ({format_size(zp.stat().st_size)})")

[INFO]  Output directory ready: /content/intersection_safety_trigger_project/outputs/tables
[INFO]  Output directory ready: /content/intersection_safety_trigger_project/outputs/figures
[INFO]  Output directory ready: /content/intersection_safety_trigger_project/outputs/samples
[INFO]  Output directory ready: /content/intersection_safety_trigger_project/outputs/logs
[INFO]  Found 4 zip file(s) in: /content/drive/MyDrive/ITS_Intersection_USDOT

Zip files found:
  Training Data 1.zip  (151.06 GB)
  Training Data 2.zip  (150.50 GB)
  Training Data 3.zip  (152.57 GB)
  Training Data 4.zip  (149.71 GB)


---
## 3 · Discover Radar Files Across All Zips

Scans the central directory of every zip — **no extraction**.

In [11]:
print("Scanning zip files for radar sensor JSON files ...")
print("(central directory only — no extraction)\n")

all_radar_files: list[dict] = []

for zip_path in zip_paths:
    print(f"  {zip_path.name} ...", end=" ", flush=True)
    files = find_radar_files_in_zip(zip_path)
    all_radar_files.extend(files)
    print(f"{len(files)} radar files")

radar_inv_df = pd.DataFrame(all_radar_files)

print(f"\n[OK] Total radar JSON files: {len(all_radar_files):,}")
print(f"     Unique runs with radar : {radar_inv_df['run_id'].nunique():,}")
print(f"     Sensor IDs found       : {sorted(radar_inv_df['sensor_id'].unique())}")
print(f"     File sizes (KB) — min/median/max: "
      f"{radar_inv_df['file_size_kb'].min():.1f} / "
      f"{radar_inv_df['file_size_kb'].median():.1f} / "
      f"{radar_inv_df['file_size_kb'].max():.1f}")

out_path = TABLES_DIR / "radar_file_inventory.csv"
radar_inv_df.to_csv(out_path, index=False)
print(f"\n[SAVED] {out_path}  (shape: {radar_inv_df.shape})")
radar_inv_df.head(8)

Scanning zip files for radar sensor JSON files ...
(central directory only — no extraction)

  Training Data 1.zip ... 416 radar files
  Training Data 2.zip ... 396 radar files
  Training Data 3.zip ... 400 radar files
  Training Data 4.zip ... 352 radar files

[OK] Total radar JSON files: 1,564
     Unique runs with radar : 391
     Sensor IDs found       : [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
     File sizes (KB) — min/median/max: 0.6 / 1118.1 / 3762.8

[SAVED] /content/intersection_safety_trigger_project/outputs/tables/radar_file_inventory.csv  (shape: (1564, 5))


,zip_name,internal_path,run_id,sensor_id,file_size_kb
0,Training Data 1.zip,Run_166/Radars_Run_166_sensor1.json,0166,1,1218.9
1,Training Data 1.zip,Run_166/Radars_Run_166_sensor4.json,0166,4,1132.6
2,Training Data 1.zip,Run_166/Radars_Run_166_sensor2.json,0166,2,1278.6
3,Training Data 1.zip,Run_166/Radars_Run_166_sensor3.json,0166,3,0.6
4,Training Data 1.zip,Run_844/Radars_Run_844_sensor2.json,0844,2,832.0
5,Training Data 1.zip,Run_844/Radars_Run_844_sensor3.json,0844,3,841.8
6,Training Data 1.zip,Run_844/Radars_Run_844_sensor1.json,0844,1,1610.2
7,Training Data 1.zip,Run_844/Radars_Run_844_sensor4.json,0844,4,1473.7


---
## 4 · Select Sample Runs

Select `MAX_RUNS_TO_INSPECT` runs that each have all 4 sensor files,
preferring runs from different zip files.

In [12]:
# Build mapping: (zip_name, run_id) -> sorted list of sensor_ids
run_sensors: dict[tuple, list] = defaultdict(list)
for _, row in radar_inv_df.iterrows():
    run_sensors[(row["zip_name"], row["run_id"])].append(int(row["sensor_id"]))
for k in run_sensors:
    run_sensors[k] = sorted(run_sensors[k])

# Mapping: zip_name -> full Path
zip_path_map = {zp.name: zp for zp in zip_paths}

# Find all runs with all MAX_SENSOR_FILES_PER_RUN sensors
full_runs = [
    (zn, rid)
    for (zn, rid), sensors in run_sensors.items()
    if len(sensors) >= MAX_SENSOR_FILES_PER_RUN
]
print(f"Runs with all {MAX_SENSOR_FILES_PER_RUN} sensor files: {len(full_runs)}")

# Select runs from different zips where possible
selected: list[tuple] = []
seen_zips: set = set()
for zn, rid in full_runs:
    if len(selected) >= MAX_RUNS_TO_INSPECT:
        break
    if zn not in seen_zips:
        selected.append((zn, rid))
        seen_zips.add(zn)
# Fill remaining slots from any zip
for zn, rid in full_runs:
    if len(selected) >= MAX_RUNS_TO_INSPECT:
        break
    if (zn, rid) not in selected:
        selected.append((zn, rid))

print(f"\nSelected {len(selected)} run(s) for schema inspection:")
for zn, rid in selected:
    sensors = run_sensors[(zn, rid)]
    print(f"  Run {rid}  |  {zn}  |  sensors: {sensors}")

Runs with all 4 sensor files: 391

Selected 3 run(s) for schema inspection:
  Run 0166  |  Training Data 1.zip  |  sensors: [1, 2, 3, 4]
  Run 0324  |  Training Data 2.zip  |  sensors: [1, 2, 3, 4]
  Run 0089  |  Training Data 3.zip  |  sensors: [1, 2, 3, 4]


---
## 5 · Radar JSON Schema Inspection

For each sampled run and sensor, read the JSON file directly from inside the zip
and inspect its structure — top-level type, record count, payload keys,
timestamp fields, and object detection lists.

In [13]:
# Central accumulator — keyed by (zip_name, run_id, sensor_id)
all_results: dict[tuple, dict] = {}

for zip_name, run_id in selected:
    zip_path = zip_path_map[zip_name]
    sensors  = run_sensors[(zip_name, run_id)][:MAX_SENSOR_FILES_PER_RUN]

    print(f"\n{'='*66}")
    print(f"  Run {run_id}  |  {zip_name}")
    print(f"{'='*66}")

    for sensor_id in sensors:
        mask = (
            (radar_inv_df["zip_name"]   == zip_name) &
            (radar_inv_df["run_id"]     == run_id)   &
            (radar_inv_df["sensor_id"]  == sensor_id)
        )
        rows = radar_inv_df[mask]
        if rows.empty:
            print(f"  [SKIP] sensor{sensor_id} — not in inventory")
            continue

        internal_path = rows.iloc[0]["internal_path"]
        file_size_kb  = rows.iloc[0]["file_size_kb"]

        print(f"\n  -- sensor{sensor_id}  ({file_size_kb:.1f} KB) ----------")
        print(f"     {internal_path}")

        result = {
            "zip_name":           zip_name,
            "run_id":             run_id,
            "sensor_id":          sensor_id,
            "internal_path":      internal_path,
            "file_size_kb":       file_size_kb,
            "top_level_type":     None,
            "num_records":        0,
            "top_level_keys":     [],
            "payload_encoding":   None,
            "payload_type":       None,
            "payload_keys":       [],
            "object_list_field":  None,
            "object_count_sample": 0,
            "sample_objects":     [],
            "timestamp_fields":   {},
            "all_object_keys":    Counter(),
            "error":              None,
        }

        data = safe_json_load_from_zip(zip_path, internal_path)
        if data is None:
            result["error"] = "failed to load"
            all_results[(zip_name, run_id, sensor_id)] = result
            print("     [ERROR] Could not load JSON")
            continue

        # ── Top-level type ───────────────────────────────────────────────────
        if isinstance(data, list):
            result["top_level_type"] = "list"
            result["num_records"]    = len(data)
            print(f"     top-level: list  ({len(data):,} records)")

            sample_records = [r for r in data[:MAX_RECORDS_PER_FILE_TO_PRINT] if isinstance(r, dict)]
            if sample_records:
                result["top_level_keys"] = list(sample_records[0].keys())
                print(f"     record keys   : {result['top_level_keys']}")

            for rec_idx, record in enumerate(sample_records):
                print(f"\n     record[{rec_idx}]:")

                # Top-level timestamp fields
                top_ts = find_timestamp_fields(record, max_depth=0)
                for ts_k, ts_v in top_ts.items():
                    fmt = guess_timestamp_format(ts_v)
                    print(f"       {ts_k}: {str(ts_v)[:60]}  [{fmt}]")
                    result["timestamp_fields"][ts_k] = str(ts_v)

                # Decode payload
                raw_payload = record.get("payload")
                if raw_payload is None:
                    print("       payload: (missing)")
                    continue

                payload, encoding = decode_payload_if_needed(raw_payload)
                result["payload_encoding"] = encoding
                print(f"       payload encoding: {encoding}")

                if not isinstance(payload, dict):
                    result["payload_type"] = type(payload).__name__
                    print(f"       payload type after decode: {result['payload_type']}")
                    if isinstance(payload, str):
                        print(f"       payload preview: {payload[:120]}")
                    continue

                result["payload_type"] = "dict"
                result["payload_keys"] = list(payload.keys())
                print(f"       payload keys  : {result['payload_keys']}")

                # Timestamps inside payload
                payload_ts = find_timestamp_fields(payload)
                for ts_path, ts_v in payload_ts.items():
                    fmt = guess_timestamp_format(ts_v)
                    full_path = f"payload.{ts_path}"
                    print(f"       {full_path}: {str(ts_v)[:50]}  [{fmt}]")
                    result["timestamp_fields"][full_path] = str(ts_v)

                # Object list
                obj_list  = extract_object_list(payload)
                obj_field = get_object_list_field_name(payload)
                if obj_list is not None:
                    result["object_list_field"]   = obj_field
                    result["object_count_sample"] = len(obj_list)
                    print(f"       object field  : '{obj_field}'  ({len(obj_list)} objects)")
                    for obj_i, obj in enumerate(obj_list[:MAX_OBJECTS_PER_RECORD_TO_PRINT]):
                        if isinstance(obj, dict):
                            result["all_object_keys"].update(obj.keys())
                            result["sample_objects"].append(obj)
                            print(f"         object[{obj_i}] keys: {list(obj.keys())}")
                            for fld in ["id","class","classification","speed","heading",
                                        "position_front","closest_lane","within_zone"]:
                                if fld in obj:
                                    print(f"           {fld}: {str(obj[fld])[:80]}")
                else:
                    print("       (no object list found — showing payload structure)")
                    for pk, pv in list(payload.items())[:6]:
                        print(f"         payload.{pk}: {type(pv).__name__} = {str(pv)[:80]}")

        elif isinstance(data, dict):
            result["top_level_type"] = "dict"
            result["top_level_keys"] = list(data.keys())
            print(f"     top-level: dict")
            print(f"     keys: {result['top_level_keys']}")
            # Look for a nested records list
            for key in ("messages", "records", "data", "frames", "detections", "events"):
                if key in data and isinstance(data[key], list):
                    result["num_records"] = len(data[key])
                    print(f"     '{key}' list: {len(data[key]):,} records")
                    break
            top_ts = find_timestamp_fields(data, max_depth=1)
            result["timestamp_fields"].update({k: str(v) for k, v in top_ts.items()})
        else:
            result["top_level_type"] = type(data).__name__
            print(f"     top-level: unexpected type {result['top_level_type']}")

        all_results[(zip_name, run_id, sensor_id)] = result

print(f"\n[OK] Schema inspection complete — {len(all_results)} sensor file(s) inspected.")


  Run 0166  |  Training Data 1.zip

  -- sensor1  (1218.9 KB) ----------
     Run_166/Radars_Run_166_sensor1.json
     top-level: list  (1,681 records)
     record keys   : ['topic', 'payload', 'qos', 'receivedAt', 'retain']

     record[0]:
       receivedAt: 2024-02-16 09:53:40  [date string]
       payload encoding: dict_or_list
       payload keys  : ['cycle_time', 'objects', 'reference_name', 'timestamp']
       payload.timestamp: 2024-02-16T14:55:51+00:00  [ISO-8601 string (timezone-aware)]
       (no object list found — showing payload structure)
         payload.cycle_time: float = 0.10000299662351608
         payload.objects: list = []
         payload.reference_name: str = sensor1
         payload.timestamp: str = 2024-02-16T14:55:51+00:00

     record[1]:
       receivedAt: 2024-02-16 09:53:40  [date string]
       payload encoding: dict_or_list
       payload keys  : ['elevation_misalignment', 'elevation_misalignment_angle', 'interference', 'rain', 'reference_name', 'roll_

---
## 6 · Object Field Availability Analysis

For every sampled detection object, check which named fields are present
and record an example value.

In [14]:
# Aggregate field counts and example values across all sampled objects
field_seen_counts: Counter = Counter()
field_examples: dict[str, tuple[str, str]] = {}  # field -> (example_value, source_path)

for (zip_name, run_id, sensor_id), result in all_results.items():
    if result.get("error"):
        continue
    source_path = f"{zip_name} / Run_{run_id} / sensor{sensor_id}"
    for obj in result.get("sample_objects", []):
        if not isinstance(obj, dict):
            continue
        for field in OBJECT_FIELDS_TO_CHECK:
            if field in obj and obj[field] is not None:
                field_seen_counts[field] += 1
                if field not in field_examples:
                    field_examples[field] = (str(obj[field])[:100], source_path)

# Build summary DataFrame
total_objects = sum(
    len(r.get("sample_objects", []))
    for r in all_results.values()
    if not r.get("error")
)

avail_rows = []
for field in OBJECT_FIELDS_TO_CHECK:
    times_seen = field_seen_counts.get(field, 0)
    ex_val, ex_path = field_examples.get(field, ("", ""))
    avail_rows.append({
        "field_name":    field,
        "times_seen":    times_seen,
        "pct_of_sample": f"{100 * times_seen / max(total_objects, 1):.0f}%",
        "example_value": ex_val,
        "example_path":  ex_path,
        "notes": "FOUND" if times_seen > 0 else "not found in sample",
    })

avail_df = pd.DataFrame(avail_rows).sort_values("times_seen", ascending=False)

out_path = TABLES_DIR / "radar_schema_field_availability.csv"
avail_df.to_csv(out_path, index=False)
print(f"[SAVED] {out_path}")
print(f"\nTotal sampled objects : {total_objects}")
print(f"\n{'Field':<28s} {'Seen':>6s}  {'%':>5s}  {'Example value (truncated)'}")
print("-" * 80)
for _, row in avail_df.iterrows():
    marker = "  *" if row["times_seen"] > 0 else ""
    print(f"  {row['field_name']:<26s} {row['times_seen']:>6d}  {row['pct_of_sample']:>5s}  "
          f"{row['example_value'][:35]}{marker}")

[SAVED] /content/intersection_safety_trigger_project/outputs/tables/radar_schema_field_availability.csv

Total sampled objects : 4

Field                          Seen      %  Example value (truncated)
--------------------------------------------------------------------------------
  id                              4   100%  243  *
  class                           4   100%  undefined  *
  heading                         4   100%  161.2992401123047  *
  speed                           4   100%  0.15457279980182648  *
  length                          4   100%  4.599999904632568  *
  position_front                  4   100%  [-142.9612274169922, 14.68905830383  *
  within_zone                     4   100%  ['zoneD']  *
  closest_lane                    4   100%  lane1  *
  position_facing                 4   100%  [-138.6040802001953, 13.21417999267  *
  tracking_status                 4   100%  {'cycles_since_last_update': 0, 'mi  *
  type                            0     0%  
  label 

---
## 7 · Position Field Probe

Inspect the structure of all position-related fields to determine which one
contains usable x/y/z coordinates.

In [15]:
POSITION_FIELDS = [
    "position_front", "position_facing", "position", "x", "y", "z"
]

pos_rows = []

for (zip_name, run_id, sensor_id), result in all_results.items():
    if result.get("error"):
        continue
    for obj in result.get("sample_objects", []):
        if not isinstance(obj, dict):
            continue
        row: dict = {
            "zip_name":  zip_name,
            "run_id":    run_id,
            "sensor_id": sensor_id,
        }
        for field in POSITION_FIELDS:
            val = obj.get(field)
            row[f"{field}__present"] = val is not None
            row[f"{field}__type"]    = type(val).__name__ if val is not None else ""
            row[f"{field}__sample"]  = str(val)[:120] if val is not None else ""
            if isinstance(val, dict):
                row[f"{field}__has_x"] = "x" in val
                row[f"{field}__has_y"] = "y" in val
                row[f"{field}__has_z"] = "z" in val
                row[f"{field}__keys"]  = str(list(val.keys())[:10])
            else:
                row[f"{field}__has_x"] = False
                row[f"{field}__has_y"] = False
                row[f"{field}__has_z"] = False
                row[f"{field}__keys"]  = ""
        pos_rows.append(row)

if pos_rows:
    pos_df = pd.DataFrame(pos_rows)
    out_path = TABLES_DIR / "radar_position_field_probe.csv"
    pos_df.to_csv(out_path, index=False)
    print(f"[SAVED] {out_path}")

    print(f"\nPosition field presence summary ({len(pos_rows)} sampled objects):")
    print(f"{'Field':<22s} {'Present':>8s}  {'Type':>12s}  {'Has x/y/z'}")
    print("-" * 65)
    for field in POSITION_FIELDS:
        col_p = f"{field}__present"
        col_t = f"{field}__type"
        col_x = f"{field}__has_x"
        if col_p in pos_df.columns:
            n_present = int(pos_df[col_p].sum())
            most_type = pos_df[col_t].mode().iloc[0] if n_present > 0 else ""
            n_has_x   = int(pos_df[col_x].sum()) if col_x in pos_df.columns else 0
            print(f"  {field:<20s} {n_present:>8d}  {most_type:>12s}  {n_has_x} have x/y/z")

    print("\nSample position values (first 3 objects):")
    for field in POSITION_FIELDS:
        col_s = f"{field}__sample"
        if col_s in pos_df.columns:
            samples = pos_df[col_s][pos_df[f"{field}__present"]].head(3).tolist()
            if samples:
                print(f"  {field}: {samples[0][:100]}")
else:
    pos_df = pd.DataFrame()
    print("[WARN] No sample objects found — position probe skipped.")

[SAVED] /content/intersection_safety_trigger_project/outputs/tables/radar_position_field_probe.csv

Position field presence summary (4 sampled objects):
Field                   Present          Type  Has x/y/z
-----------------------------------------------------------------
  position_front              4          list  0 have x/y/z
  position_facing             4          list  0 have x/y/z
  position                    0                0 have x/y/z
  x                           0                0 have x/y/z
  y                           0                0 have x/y/z
  z                           0                0 have x/y/z

Sample position values (first 3 objects):
  position_front: [-142.9612274169922, 14.689058303833008, 0.0]
  position_facing: [-138.6040802001953, 13.214179992675781, 0.0]


---
## 8 · Timestamp Probe

Collect all timestamp-like field values found during schema inspection
and determine which format they use.

In [16]:
ts_rows = []

for (zip_name, run_id, sensor_id), result in all_results.items():
    if result.get("error"):
        continue
    for ts_path, ts_raw in result.get("timestamp_fields", {}).items():
        # Re-guess format from the stored string
        fmt = guess_timestamp_format(ts_raw)
        note = ""
        if ts_path == "receivedAt":
            note = "top-level MQTT receipt time — primary alignment candidate"
        elif "timestamp" in ts_path.lower():
            note = "payload timestamp — compare with receivedAt for latency"
        ts_rows.append({
            "zip_name":       zip_name,
            "run_id":         run_id,
            "sensor_id":      sensor_id,
            "field_path":     ts_path,
            "sample_value":   str(ts_raw)[:80],
            "guessed_format": fmt,
            "alignment_note": note,
        })

if ts_rows:
    ts_df = pd.DataFrame(ts_rows)
    out_path = TABLES_DIR / "radar_timestamp_probe.csv"
    ts_df.to_csv(out_path, index=False)
    print(f"[SAVED] {out_path}")

    print("\nTimestamp fields discovered (unique field paths):")
    print(f"{'Field path':<35s} {'Format':<35s} {'Sample value'}")
    print("-" * 110)
    for _, row in ts_df.drop_duplicates("field_path").iterrows():
        print(f"  {row['field_path']:<33s} {row['guessed_format']:<35s} {row['sample_value'][:35]}")

    print("\nAlignment recommendation:")
    alignment_candidates = ts_df[ts_df["alignment_note"] != ""]
    if not alignment_candidates.empty:
        for _, row in alignment_candidates.drop_duplicates("field_path").iterrows():
            print(f"  '{row['field_path']}' — {row['alignment_note']}")
    else:
        print("  No clear candidate identified yet — inspect the output CSV.")
else:
    ts_df = pd.DataFrame()
    print("[WARN] No timestamp fields found — check schema inspection output above.")

[SAVED] /content/intersection_safety_trigger_project/outputs/tables/radar_timestamp_probe.csv

Timestamp fields discovered (unique field paths):
Field path                          Format                              Sample value
--------------------------------------------------------------------------------------------------------------
  receivedAt                        date string                         2024-02-16 09:53:40
  payload.timestamp                 ISO-8601 string (timezone-aware)    2024-02-16T14:52:45+00:00

Alignment recommendation:
  'receivedAt' — top-level MQTT receipt time — primary alignment candidate
  'payload.timestamp' — payload timestamp — compare with receivedAt for latency


---
## 9 · Sensor-to-Sensor Schema Comparison

Compare sensor1 through sensor4 across sampled runs to check whether
all sensors share the same field structure.

In [17]:
comp_rows = []

for (zip_name, run_id, sensor_id), result in all_results.items():
    all_obj_keys = result.get("all_object_keys", Counter())

    def has(*fields):
        return any(all_obj_keys.get(f, 0) > 0 for f in fields)

    comp_rows.append({
        "zip_name":              zip_name,
        "run_id":                run_id,
        "sensor_id":             sensor_id,
        "num_records":           result.get("num_records", 0),
        "top_level_type":        result.get("top_level_type", ""),
        "payload_encoding":      result.get("payload_encoding", ""),
        "has_payload_dict":      result.get("payload_type") == "dict",
        "object_list_field":     result.get("object_list_field", ""),
        "has_objects":           result.get("object_list_field") is not None,
        "object_count_sample":   result.get("object_count_sample", 0),
        "has_id":                has("id", "object_id", "track_id"),
        "has_class":             has("class", "classification", "type", "label"),
        "has_speed":             has("speed", "velocity", "speed_magnitude"),
        "has_heading":           has("heading", "course", "orientation"),
        "has_position_front":    has("position_front"),
        "has_position_facing":   has("position_facing"),
        "has_position_any":      has("position", "position_front", "position_facing", "x", "y"),
        "has_lane":              has("closest_lane", "lane", "lane_id"),
        "has_zone":              has("within_zone", "zone", "zone_id"),
        "has_tracking_status":   has("tracking_status", "tracking_quality", "confidence"),
        "error":                 result.get("error", ""),
    })

comp_df = pd.DataFrame(comp_rows)
out_path = TABLES_DIR / "radar_sensor_schema_comparison.csv"
comp_df.to_csv(out_path, index=False)
print(f"[SAVED] {out_path}")
print()

# Print a readable bool table
bool_cols = [c for c in comp_df.columns if c.startswith("has_")]
display_cols = ["run_id", "sensor_id", "num_records", "object_count_sample"] + bool_cols
display(comp_df[display_cols].fillna(""))

[SAVED] /content/intersection_safety_trigger_project/outputs/tables/radar_sensor_schema_comparison.csv



,run_id,sensor_id,num_records,object_count_sample,has_payload_dict,has_objects,has_id,has_class,has_speed,has_heading,has_position_front,has_position_facing,has_position_any,has_lane,has_zone,has_tracking_status
0,0166,1,1681,0,True,False,False,False,False,False,False,False,False,False,False,False
1,0166,2,1680,1,True,True,True,True,True,True,True,True,True,True,True,True
2,0166,3,2,0,True,False,False,False,False,False,False,False,False,False,False,False
3,0166,4,1682,0,True,False,False,False,False,False,False,False,False,False,False,False
4,0324,1,1867,0,True,False,False,False,False,False,False,False,False,False,False,False
5,0324,2,1868,2,True,True,True,True,True,True,True,True,True,True,True,True
6,0324,3,2,0,True,False,False,False,False,False,False,False,False,False,False,False
7,0324,4,1870,0,True,False,False,False,False,False,False,False,False,False,False,False
8,0089,1,1744,0,True,False,False,False,False,False,False,False,False,False,False,False
9,0089,2,1750,1,True,True,True,True,True,True,True,True,True,True,True,True


---
## 10 · Auto-Generated Schema Findings Summary

Generates a markdown report from the accumulated results and saves it to
`outputs/tables/notebook_01_radar_schema_findings.md`.

In [19]:
import datetime

# ── Aggregate facts from all results ─────────────────────────────────────────
n_inspected     = len(all_results)
n_errors        = sum(1 for r in all_results.values() if r.get('error'))
n_ok            = n_inspected - n_errors

good_results    = [r for r in all_results.values() if not r.get('error')]

top_types       = Counter(r['top_level_type'] for r in good_results if r['top_level_type'])
pay_encodings   = Counter(r['payload_encoding'] for r in good_results if r['payload_encoding'])
pay_types       = Counter(r['payload_type'] for r in good_results if r['payload_type'])
obj_fields_used = Counter(r['object_list_field'] for r in good_results if r['object_list_field'])

# Field availability from comp_df
bool_cols = [c for c in comp_df.columns if c.startswith('has_')]
field_pct = {}
for col in bool_cols:
    # Only calculate for non-error rows
    valid_mask = comp_df['error'].isna() | (comp_df['error'] == '')
    n_valid = valid_mask.sum()
    if n_valid > 0:
        field_pct[col] = f"{100 * comp_df.loc[valid_mask, col].sum() / n_valid:.0f}%"

# Timestamps
ts_paths = list({r_ts for r in good_results for r_ts in r.get('timestamp_fields', {})})

# Object count stats
obj_counts = [r['object_count_sample'] for r in good_results if r['object_count_sample'] > 0]
obj_count_str = (
    f"min={min(obj_counts)}, median={sorted(obj_counts)[len(obj_counts)//2]}, max={max(obj_counts)}"
    if obj_counts else "(no objects found)"
)

# ── Build markdown ────────────────────────────────────────────────────────────
L = []
def ln(s=""): L.append(s)

ln("# Notebook 01 — Radar JSON Schema Findings")
ln("**Physics-Informed Trigger Event Analysis — Urban Intersection Safety**  ")
ln(f"**Generated:** {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}  ")
ln(f"**Runs inspected:** {MAX_RUNS_TO_INSPECT}  ")
ln(f"**Sensor files read:** {n_inspected} ({n_ok} OK, {n_errors} errors)  ")
ln()
ln("---")
ln()
ln("## 1. Top-Level Structure")
ln()
ln(f"Top-level JSON type seen: {dict(top_types)}")
ln()
if top_types.get("list", 0) > 0:
    ln("The radar sensor JSON files are a **list of records** (same pattern as trigger JSON).")
    ln("Each element in the list is an object with keys such as:")
    sample_keys = good_results[0].get("top_level_keys", []) if good_results else []
    ln(f"  `{sample_keys}`")
    ln()
    ln("This is consistent with an **MQTT message log** format.")
elif top_types.get("dict", 0) > 0:
    ln("The radar sensor JSON files are a **single dict** (not a list).")
    sample_keys = good_results[0].get("top_level_keys", []) if good_results else []
    ln(f"  Top-level keys: `{sample_keys}`")
ln()
ln("---")
ln()
ln("## 2. Payload Encoding")
ln()
ln(f"Payload encoding found: {dict(pay_encodings)}  ")
ln(f"Payload type after decode: {dict(pay_types)}")
ln()
if pay_encodings.get("dict_or_list", 0) > 0:
    ln("`payload` is already a Python dict — no additional decoding needed.")
elif pay_encodings.get("string_json", 0) > 0:
    ln("`payload` is a JSON-encoded **string** — must be parsed with `json.loads()` before use.")
elif pay_encodings.get("base64_json", 0) > 0:
    ln("`payload` is **base64-encoded JSON** — must be base64-decoded then JSON-parsed.")
elif pay_encodings.get("plain_string", 0) > 0:
    ln("`payload` is a plain string with no recognisable JSON or base64 structure.")
ln()
ln("---")
ln()
ln("## 3. Where Object Detections Are Stored")
ln()
if obj_fields_used:
    ln(f"Object list field name(s) found: {dict(obj_fields_used)}")
    primary_field = obj_fields_used.most_common(1)[0][0]
    ln(f"  → Primary field to use: **`payload.{primary_field}`**")
    ln(f"  → Object counts per record sample: {obj_count_str}")
else:
    ln("No object list field was found in the sampled files.  ")
    ln("Possible reasons: empty payload, different field name, or zero-object records.  ")
    ln("Inspect the 'payload structure' printout in cell 5 manually.")
ln()
ln("---")
ln()
ln("## 4. Timestamp Fields")
ln()
if ts_paths:
    ln(f"Timestamp fields found: {ts_paths}")
    ln()
    if "receivedAt" in ts_paths:
        ln("**`receivedAt`** is present at the top level of each MQTT record.")
        ln("This is the **recommended alignment field** for matching radar records")
        ln("with trigger `receivedAt` timestamps in Notebook 03.")
    payload_ts = [p for p in ts_paths if p.startswith("payload.")]
    if payload_ts:
        ln(f"  Payload timestamps also found: {payload_ts}")
        ln("  These may represent the actual sensor capture time (before network transmission).")
        ln("  Use the difference between `receivedAt` and `payload.timestamp` to estimate latency.")
else:
    ln("No timestamp fields were found — check cell 5 output manually.")
ln()
ln("---")
ln()
ln("## 5. Object Field Availability")
ln()
ln(f"Total sampled objects: {total_objects}")
ln()
ln("| Field | % present | Notes |")
ln("|-------|-----------|-------|")
for col in bool_cols:
    pct  = field_pct.get(col, "?")
    name = col.replace("has_", "")
    note = ""
    if name == "speed":          note = "key for physics features"
    elif name == "heading":      note = "key for physics features"
    elif name == "position_any": note = "key for distance / TTC proxy"
    elif name == "lane":         note = "key for lane-level trigger matching"
    elif name == "zone":         note = "key for zone-level trigger matching"
    elif name == "class":        note = "key for object type filtering"
    elif name == "id":           note = "key for object tracking across frames"
    ln(f"| `{name}` | {pct} | {note} |")
ln()
ln("---")
ln()
ln("## 6. Position Field Assessment")
ln()
if not pos_df.empty:
    for field in ["position_front", "position_facing", "position"]:
        col_p = f"{field}__present"
        col_t = f"{field}__type"
        col_x = f"{field}__has_x"
        if col_p in pos_df.columns:
            n_p = int(pos_df[col_p].sum())
            n_x = int(pos_df[col_x].sum()) if col_x in pos_df.columns else 0
            ln(f"- `{field}`: present in {n_p} objects; {n_x} have x/y/z sub-fields")
    ln()
    ln("**Recommended position field for feature extraction:** inspect the sample values")
    ln("printed in cell 7 and choose the field that consistently contains x/y coordinates.")
else:
    ln("Position probe was skipped (no sampled objects). Inspect cell 7 output manually.")
ln()
ln("---")
ln()
ln("## 7. Physics Feature Readiness")
ln()
ln("Based on field availability from the sample:")
ln()
features = [
    ("Object count",           field_pct.get("has_id", "?"),      "count objects per time window"),
    ("Speed stats",            field_pct.get("has_speed", "?"),   "mean/max speed in window"),
    ("Heading stats",          field_pct.get("has_heading", "?"), "heading spread, alignment"),
    ("Position / distance",    field_pct.get("has_position_any", "?"), "inter-object distance, TTC proxy"),
    ("Lane activity",          field_pct.get("has_lane", "?"),    "count objects per lane"),
    ("Zone activity",          field_pct.get("has_zone", "?"),    "count objects per zone"),
    ("Object classification",  field_pct.get("has_class", "?"),   "filter by vehicle type"),
    ("Tracking quality",       field_pct.get("has_tracking_status", "?"), "filter by confidence"),
]
ln("| Feature | Field availability | Purpose |")
ln("|---------|-------------------|---------|")
for feat, pct, purpose in features:
    ln(f"| {feat} | {pct} | {purpose} |")
ln()
ln("**TTC proxy** (time-to-collision): requires position + speed → computable if both present.")
ln("**Stopping distance proxy**: requires speed → computable.")
ln("**Kinetic energy proxy**: requires speed + (length×width as mass proxy) → likely computable.")
ln("**PDE density/flow residual**: requires object count + position → likely computable.")
ln()
ln("---")
ln()
ln("## 8. Sensor Consistency")
ln()
if len(comp_df) > 0:
    n_sensors = len(comp_df)
    # Fix: all() needs an iterable
    identical_structure = all([
        comp_df["top_level_type"].nunique() <= 1
    ])
    ln(f"Sensors inspected: {n_sensors} (across {MAX_RUNS_TO_INSPECT} run(s))")
    ln(f"All sensors share the same top-level type: {yes_no(identical_structure)}")
    ln()
    ln("See `radar_sensor_schema_comparison.csv` for the full per-sensor breakdown.")
ln()
ln("---")
ln()
ln("## 9. Uncertainties and Open Questions")
ln()
ln("- [ ] Are the object coordinates in a global (lat/lon) or local (metres from sensor) system?")
ln("- [ ] What is the coordinate origin and orientation for each sensor?")
ln("- [ ] How are the 4 sensors spatially arranged at the intersection?")
ln("- [ ] Is `receivedAt` consistent enough for millisecond-level alignment with trigger files?")
ln("- [ ] Do all records in a file have the same payload structure, or does it vary?")
ln("- [ ] Is the object `id` stable across consecutive MQTT records (tracklet)?")
ln("- [ ] Does `within_zone` or `closest_lane` directly correspond to trigger zone/lane identifiers?")
ln()
ln("---")
ln()
ln("## 10. Next Steps — Notebook 02")
ln()
ln("**Notebook 02: Trigger Log Deep Inspection** should:")
ln("- Explicitly decode the `payload` field of trigger MQTT messages.")
ln("- Print the `topic` strings (10–20 samples) to extract lane/zone semantics.")
ln("- Find `reference_name`, `associated_lane`, `associated_zone` inside `payload`.")
ln("- Build a `reference_name → lane/zone/sensor` mapping table.")
ln("- Cross-reference trigger `receivedAt` timestamps with radar record timestamps")
ln("  from matching runs to measure the alignment gap.")
ln("- Confirm that `closest_lane` / `within_zone` in radar objects maps to")
ln("  the same lane/zone identifiers used in trigger messages.")

summary_text = "\n".join(L)
print(summary_text[:3000], "...\n[truncated for display]")

out_path = TABLES_DIR / "notebook_01_radar_schema_findings.md"
out_path.write_text(summary_text, encoding="utf-8")
print(f"\n[SAVED] {out_path}")

# Notebook 01 — Radar JSON Schema Findings
**Physics-Informed Trigger Event Analysis — Urban Intersection Safety**  
**Generated:** 2026-05-14 06:37  
**Runs inspected:** 3  
**Sensor files read:** 12 (12 OK, 0 errors)  

---

## 1. Top-Level Structure

Top-level JSON type seen: {'list': 12}

The radar sensor JSON files are a **list of records** (same pattern as trigger JSON).
Each element in the list is an object with keys such as:
  `['topic', 'payload', 'qos', 'receivedAt', 'retain']`

This is consistent with an **MQTT message log** format.

---

## 2. Payload Encoding

Payload encoding found: {'dict_or_list': 12}  
Payload type after decode: {'dict': 12}

`payload` is already a Python dict — no additional decoding needed.

---

## 3. Where Object Detections Are Stored

Object list field name(s) found: {'objects': 3}
  → Primary field to use: **`payload.objects`**
  → Object counts per record sample: min=1, median=1, max=2

---

## 4. Timestamp Fields

Timestamp fields found: ['payl

---
## 11 · Package Outputs and Download

In [20]:
import shutil, datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_stem  = f"task1_outputs_{timestamp}"
zip_path  = Path(f"/content/{zip_stem}.zip")

shutil.make_archive(
    str(zip_path.with_suffix("")),
    "zip",
    str(OUTPUTS_DIR),
)

print(f"[OK] Archive: {zip_path}  ({zip_path.stat().st_size / 1024:.1f} KB)")
print()
print("Contents included:")
for f in sorted(TABLES_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")

[OK] Archive: /content/task1_outputs_20260514_063733.zip  (31.3 KB)

Contents included:
  notebook_01_radar_schema_findings.md  (4.8 KB)
  radar_file_inventory.csv  (106.6 KB)
  radar_position_field_probe.csv  (1.8 KB)
  radar_schema_field_availability.csv  (1.7 KB)
  radar_sensor_schema_comparison.csv  (1.7 KB)
  radar_timestamp_probe.csv  (3.5 KB)
  sample_trigger_semantics_probe.csv  (0.7 KB)
  training_zip_inventory.csv  (4.7 KB)
  training_zip_sample_paths.csv  (66.9 KB)


In [21]:
try:
    from google.colab import files
    print(f"[INFO] Starting download: {zip_path.name}")
    files.download(str(zip_path))
    print("[OK]  Download initiated.")
except ImportError:
    print(f"[INFO] Not in Colab — archive is at: {zip_path}")

[INFO] Starting download: task1_outputs_20260514_063733.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[OK]  Download initiated.


---

## Notebook Complete

### Files saved

| File | Description |
|------|-------------|
| `radar_file_inventory.csv` | All radar JSON paths across all 4 zips |
| `radar_schema_field_availability.csv` | Which object fields exist and how often |
| `radar_position_field_probe.csv` | Position field types, sub-keys, sample values |
| `radar_timestamp_probe.csv` | Timestamp fields, format, alignment notes |
| `radar_sensor_schema_comparison.csv` | sensor1–sensor4 structure comparison |
| `notebook_01_radar_schema_findings.md` | Auto-generated findings summary |

### Next

**Notebook 02 — Trigger Log Deep Inspection:**
decode trigger `payload`, extract `topic` string semantics, and build the
`reference_name → lane/zone/sensor` mapping table.